# Sneaker listings: Thompson's test rejects three prices

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Outlier Test (Smirnov-Grubbs)** · a worked example from the StatNex teaching guide

**The story.** A reseller scraped 47 online listings for the same sneaker model to set a fair price. Two listings show no price, and one asks \$1,999.

**The question.** Is the \$1,999 listing an outlier, and is any other price suspicious?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `01_basic/outlier_test.csv` (47 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Listing,Price ($)
Listing 1,183.0
Listing 2,147.0
Listing 3,171.0
Listing 4,142.0
Listing 5,162.0
Listing 6,162.0
Listing 7,174.0
Listing 8,156.0
Listing 9,174.0
Listing 10,152.0
Listing 11,156.0
Listing 12,150.0
Listing 13,
Listing 14,130.0
Listing 15,174.0
Listing 16,141.0
Listing 17,166.0
Listing 18,168.0
Listing 19,180.0
Listing 20,175.0
Listing 21,149.0
Listing 22,171.0
Listing 23,179.0
Listing 24,187.0
Listing 25,158.0
Listing 26,158.0
Listing 27,169.0
Listing 28,148.0
Listing 29,173.0
Listing 30,183.0
Listing 31,1999.0
Listing 32,175.0
Listing 33,176.0
Listing 34,184.0
Listing 35,156.0
Listing 36,146.0
Listing 37,139.0
Listing 38,171.0
Listing 39,181.0
Listing 40,171.0
Listing 41,179.0
Listing 42,
Listing 43,175.0
Listing 44,144.0
Listing 45,154.0
Listing 46,133.0
Listing 47,169.0
""")

## Helper: thompson

A step that no library does, written out. In the code download it is the file `thompson.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Thompson's (1935) rejection test for one outlier. SciPy and statsmodels have no such test.

tau is how many standard deviations the candidate sits from the mean of all n values,
the standard deviation taken on n (ddof=0), and tau √(n - 2) / √(n - 1 - tau²) follows
Student's t on n - 2 df -- exactly, because it is then the candidate's externally
studentized deviation. Unlike
Smirnov-Grubbs there is no correction for having picked the most extreme value, which
is why it rejects more readily. The P value is doubled under a two-sided test.
"""

import numpy as np
from scipy import stats


def thompson(x, side="two-sided"):
    """(position of the candidate in x, tau, P). side: "two-sided", "largest" or "smallest"."""
    n = x.size
    sd = np.std(x, ddof=0)
    if sd == 0:
        return -1, 0.0, 1.0
    deviations = x - np.mean(x)
    if side == "two-sided":
        index = int(np.argmax(np.abs(deviations)))
    elif side == "largest":
        index = int(np.argmax(deviations))
    else:
        index = int(np.argmin(deviations))
    tau = abs(deviations[index]) / sd
    if n - 1 - tau**2 <= 0:
        return index, tau, 0.0
    t = tau * np.sqrt(n - 2) / np.sqrt(n - 1 - tau**2)
    sides = 2 if side == "two-sided" else 1
    return index, tau, min(1.0, sides * stats.t.sf(t, n - 2))

## Libraries

In [ ]:
import numpy as np
import pandas as pd
# thompson: defined in the helper cell thompson above (thompson.py in the code download)

pd.set_option("display.width", 140)

## The data

In [ ]:
# One column per variable. Blank and text cells are skipped but keep their row
# number, so a result can be traced back to its row.
# Every test below assumes the values other than the candidate come from one normal
# distribution. A small P says the candidate sits further out than such a sample
# would plausibly put its most extreme value; it cannot say why (a typo, a fault, a
# genuine rare case), so check the record before deleting anything, and report the
# results with and without it. Testing one value at a time can miss two outliers
# lying close together, which inflate the SD between them (masking).
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Price ($)"]
alpha = 5 / 100
# Two-sided: the candidate is whichever value lies furthest from the mean.
side = "two-sided"

## Repeated Thompson test

In [ ]:
# Test the candidate (thompson.py); if P < alpha, exclude it and test again on what is
# left, until a candidate is retained or fewer than 3 values remain. Rows are tracked
# by position, so of two equal values only the excluded one is marked.
# Thompson's test makes no allowance for having picked the most extreme of n values,
# so it flags values far more readily than Smirnov-Grubbs and, applied again and again,
# can trim genuine values from the tails.
results = {}
for name in columns:
    raw = pd.to_numeric(data[name], errors="coerce").to_numpy(float)
    alive = np.flatnonzero(~np.isnan(raw))
    total = alive.size
    removed, steps = [], []
    while alive.size >= 3:
        x = raw[alive]
        index, statistic, p = thompson(x, side)
        # When every value is the same, thompson() returns -1: nothing can be an outlier.
        if index < 0:
            break
        is_max = x[index] >= x.mean() if side == "two-sided" else side == "largest"
        rejected = p < alpha
        steps.append([len(steps) + 1, x.size, x.mean(), x.var(ddof=1), x.std(ddof=1),
                      np.nan if is_max else x.min(), x.max() if is_max else np.nan,
                      alive[index] + 1, p, "rejected" if rejected else "retained",
                      "largest is an outlier" if is_max else "smallest is an outlier"])
        if not rejected:
            break
        removed.append(alive[index])
        alive = np.delete(alive, index)

    outlier = np.isin(np.arange(raw.size), removed)
    results[name] = {
        "data": pd.DataFrame({"No.": np.arange(1, raw.size + 1), "original": raw,
                              "after exclusion": np.where(outlier, np.nan, raw),
                              "outlier": np.where(outlier, raw, np.nan)}),
        "summary": pd.Series({"original": total, "after exclusion": total - len(removed),
                              "outliers": len(removed)}),
        "trace": pd.DataFrame(steps, columns=["step", "n", "mean", "variance", "sd", "min",
                                              "max", "No.", "P", "verdict", "alternative"]),
    }
    print(f"\n=== {name} ===")
    print(results[name]["data"].to_string(index=False))
    print()
    print(results[name]["summary"].to_string())
    print()
    print(results[name]["trace"].to_string(index=False))

## What the result says

Thompson's test also looks at the most extreme price in each step, but it rejects more easily than the standard Smirnov-Grubbs test.

Step 1 rejects \$1,999, with a P value that is practically zero. A P value is the chance of a value this extreme if nothing were wrong. Step 2 rejects the smallest price, \$130 (P = 0.023), and step 3 rejects \$133 (P = 0.025). Step 4 retains \$139 with P = 0.053, just above the 5% significance level, the cut-off the test uses. In the end 42 of the 45 prices remain.

So the method matters. \$130 and \$133 are low but believable prices for this sneaker. Thompson's test is a screen that tells you which listings to check.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.